# Setup

## Environment Setup

In [1]:
%pip install numpy scipy librosa soundfile matplotlib Pillow pydub ffmpeg-python pandas ipython mp3stego-lib tqdm bitarray


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# -- Setup environment in Colab --
import os

if not os.path.exists("assets"):
    print("Fetching assets from GitHub...")
    !git clone --quiet https://github.com/Monczak/audio-steganography.git /tmp/audio-steg
    !mv /tmp/audio-steg/assets .
    !rm -rf /tmp/audio-steg
    print("Assets mounted and ready!")

In [3]:
import os
import io
import wave
import struct
import subprocess
import tempfile
import warnings
import numpy as np
import scipy.io.wavfile as wav
import librosa
import librosa.display
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Audio, display, HTML
from scipy.signal import butter, filtfilt
from PIL import Image
from scipy.signal import hilbert, butter, sosfilt
import scipy.signal
import hashlib
from mp3stego import Steganography
import ipywidgets as widgets
from IPython.display import clear_output

warnings.filterwarnings("ignore")

SR = 44100
ASSETS_DIR = Path("assets")
OUTPUT = Path("output")
AUDIO_DIR = ASSETS_DIR / "audio"
IMG_DIR = ASSETS_DIR / "img"
OUTPUT_AUDIO_DIR = OUTPUT / "audio" 

## The Meat
Implementations of the actual stego algorithms and helper functions.

In [4]:
import os
import io
import wave
import struct
import subprocess
import tempfile
import warnings
import numpy as np
import scipy.io.wavfile as wav
import librosa
import librosa.display
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Audio, display, HTML
from scipy.signal import butter, filtfilt
from PIL import Image
from scipy.signal import hilbert, butter, sosfilt
import scipy.signal
import hashlib
from mp3stego import Steganography
import ipywidgets as widgets
from IPython.display import clear_output

def snr(original, modified):
    """Signal-to-Noise Ratio in dB."""
    original = np.asarray(original, dtype=np.float64)
    modified = np.asarray(modified, dtype=np.float64)
    noise = original - modified
    power_signal = np.sum(original ** 2)
    power_noise = np.sum(noise ** 2)
    if power_noise == 0:
        return float("inf")
    return 10 * np.log10(power_signal / power_noise)

def psnr(original, modified):
    """Peak Signal-to-Noise Ratio in dB."""
    original = np.asarray(original, dtype=np.float64)
    modified = np.asarray(modified, dtype=np.float64)
    noise = original - modified
    mse = np.mean(noise ** 2)
    if mse == 0:
        return float("inf")
    peak = np.max(np.abs(original))
    return 10 * np.log10(peak ** 2 / mse)

def ber(original_bytes, recovered_bytes):
    """Bit Error Rate: fraction of mismatched bits (now unpacks bytestrings)."""
    orig_arr = np.frombuffer(original_bytes, dtype=np.uint8)
    recov_arr = np.frombuffer(recovered_bytes, dtype=np.uint8)
    
    orig_bits = np.unpackbits(orig_arr) if len(orig_arr) > 0 else np.array([])
    recov_bits = np.unpackbits(recov_arr) if len(recov_arr) > 0 else np.array([])
    
    length = min(len(orig_bits), len(recov_bits))
    if length == 0:
        return 1.0
        
    errors = np.sum(orig_bits[:length] != recov_bits[:length])
    return errors / length

def capacity_bits(n_samples, bits_per_sample=1):
    """Maximum payload capacity in bits (excluding header)."""
    return n_samples * bits_per_sample

def load_audio(file_path):
    """Load an audio file and ensures it is converted to mono."""
    audio, sr = sf.read(file_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio, sr

def text_to_bytes(text):
    """Convert a string directly to a bytestring."""
    return text.encode("utf-8")

def bytes_to_text(b):
    """Convert a bytestring back to a string."""
    return b.decode("utf-8", errors="replace")

def plot_spectrogram(audio, sr, title="Spectrogram", ax=None):
    """Display a mel spectrogram."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 4))
    S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")

def plot_waveform(audio, sr, title="Waveform", ax=None):
    """Display the time-domain waveform."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 3))
    t = np.arange(len(audio)) / sr
    ax.plot(t, audio, linewidth=0.5)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, t[-1])

def mp3_roundtrip(input_wav, bitrate="128k"):
    """WAV -> MP3 -> WAV round-trip via ffmpeg. Returns path to recovered WAV."""
    mp3_path = input_wav.replace(".wav", "_compressed.mp3")
    recovered_path = input_wav.replace(".wav", "_recovered.wav")
    subprocess.run(["ffmpeg", "-y", "-i", input_wav, "-b:a", bitrate, mp3_path], capture_output=True)
    subprocess.run(["ffmpeg", "-y", "-i", mp3_path, recovered_path], capture_output=True)
    return recovered_path

def display_audio_comparison(cover_audio, stego_audio, sr, title="Audio Comparison"):
    """Displays spectrograms and playable audio for cover vs stego."""
    print(f"\n{"="*20} {title} {"="*20}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_spectrogram(cover_audio, sr, title="Original Spectrogram", ax=axes[0])
    plot_spectrogram(stego_audio, sr, title="Stego Spectrogram", ax=axes[1])
    plt.tight_layout()
    plt.show()
    
    print("Original Audio:")
    display(Audio(data=cover_audio, rate=sr))
    print("Stego Audio:")
    display(Audio(data=stego_audio, rate=sr))

def lsb_encode(audio_data, payload_bytes, n_bits=1):
    original_shape = audio_data.shape
    # Flatten array to seamlessly process stereo/multi-channel samples
    samples = np.int16(np.asarray(audio_data).flatten() * 32767)
    
    # Header: 32 bits (4 bytes) representing the payload length in bytes
    header = len(payload_bytes).to_bytes(4, byteorder="big")
    full_payload = header + payload_bytes
    
    total_bits = len(full_payload) * 8
    samples_needed = (total_bits + n_bits - 1) // n_bits
    
    if samples_needed > len(samples):
        raise ValueError(f"Payload too large: need {samples_needed} samples, have {len(samples)}")
        
    mask = ~((1 << n_bits) - 1) & 0xFFFF
    bit_idx = 0
    stego_samples = samples.copy()
    
    for i in range(samples_needed):
        val = int(stego_samples[i]) & 0xFFFF
        val = val & mask
        
        chunk = 0
        for b in range(n_bits):
            if bit_idx < total_bits:
                byte_val = full_payload[bit_idx // 8]
                bit_val = (byte_val >> (7 - (bit_idx % 8))) & 1
                chunk = (chunk << 1) | bit_val
                bit_idx += 1
            else:
                chunk = chunk << 1
                
        val = val | chunk
        
        # Restore two's complement constraint for negative values
        if val >= 0x8000:
            val -= 0x10000
        stego_samples[i] = val
        
    # Reshape the flattened outcome back to it's original multi-channel setup
    return (stego_samples.astype(np.float64) / 32767.0).reshape(original_shape)

def lsb_decode(stego_audio, n_bits=1):
    # Flatten array to seamlessly process stereo/multi-channel samples
    samples = np.int16(np.asarray(stego_audio).flatten() * 32767)
    
    extracted_bytes = bytearray()
    current_byte = 0
    bits_in_current_byte = 0
    
    payload_len = 0
    header_parsed = False
    
    for i in range(len(samples)):
        val = int(samples[i]) & 0xFFFF
        
        for b in range(n_bits - 1, -1, -1):
            bit_val = (val >> b) & 1
            current_byte = (current_byte << 1) | bit_val
            bits_in_current_byte += 1
            
            if bits_in_current_byte == 8:
                extracted_bytes.append(current_byte)
                current_byte = 0
                bits_in_current_byte = 0
                
                # Setup payload length once the 4-byte header is assembled
                if not header_parsed and len(extracted_bytes) == 4:
                    payload_len = int.from_bytes(extracted_bytes, byteorder='big')
                    header_parsed = True
                    
                # Return successfully if we've received the entire expected payload
                if header_parsed and len(extracted_bytes) == 4 + payload_len:
                    return bytes(extracted_bytes[4:])
                    
    # Failsafe if payload ends up truncated by audio end
    return bytes(extracted_bytes[4:4 + payload_len]) if len(extracted_bytes) > 4 else b""

def test_lsb_variant(cover_audio, sr, payload_bytes, n_bits):
    """Runs an LSB encode/decode test in-memory and returns metrics."""
    stego_audio = lsb_encode(cover_audio, payload_bytes, n_bits)
    recovered_bytes = lsb_decode(stego_audio, n_bits)
    
    error_rate = ber(payload_bytes, recovered_bytes)
    quality = snr(cover_audio, stego_audio)
    cap = capacity_bits(len(cover_audio), n_bits)
    
    return {
        "Variant": f"{n_bits}-bit LSB",
        "SNR (dB)": round(quality, 2),
        "BER": round(error_rate, 4),
        "Capacity (bits)": cap,
        "Recovered Text Preview": bytes_to_text(recovered_bytes)[:50]
    }, stego_audio

def lsb_steganalysis(audio_data, n_bits=1):
    """
    Chi-squared test on LSB distribution of audio array.
    Returns: chi2_stat, p_value, lsb_ratio
    """
    samples = np.int16(audio_data * 32767)
    lsbs = [int(s) & ((1 << n_bits) - 1) for s in samples]
    n_values = 1 << n_bits
    
    observed = np.zeros(n_values)
    for lsb in lsbs:
        observed[lsb] += 1
        
    expected = np.full(n_values, len(lsbs) / n_values)
    chi2_stat = np.sum((observed - expected) ** 2 / expected)
    
    from scipy.stats import chi2
    p_value = 1 - chi2.cdf(chi2_stat, df=n_values - 1)
    lsb_ratio = np.mean([b & 1 for b in lsbs])
    
    return chi2_stat, p_value, lsb_ratio

def mp3_roundtrip_variable(audio_data, sr, bitrate="128k"):
    """WAV -> MP3 -> WAV round-trip passing data through temp files."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as wav_in, \
        tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as mp3_out, \
        tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as wav_out:
        
        sf.write(wav_in.name, audio_data, sr, subtype="PCM_16")
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", wav_in.name, "-b:a", bitrate, mp3_out.name], capture_output=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", mp3_out.name, wav_out.name], capture_output=True)
        
        recovered_audio, _ = sf.read(wav_out.name)
        
    os.unlink(wav_in.name)
    os.unlink(mp3_out.name)
    os.unlink(wav_out.name)
    
    return recovered_audio

def apply_lowpass(data, sr, cutoff=2000, order=4):
    nyq = 0.5 * sr
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def phase_coding_encode(audio_data, payload_bytes, block_size=2048):
    """
    Classical phase coding: embed payload into block 0's phase, propagate phase differences forward.
    
    Args:
        audio_data: numpy array of audio samples (float, range [-1, 1])
        payload_bytes: bytes object to hide
        block_size: FFT block length
    
    Returns:
        stego_audio: reconstructed audio with embedded payload
    """
    audio = np.asarray(audio_data, dtype=np.float64)
    n_samples = len(audio)
    n_blocks = (n_samples + block_size - 1) // block_size
    
    # Pad audio to multiple of block_size
    padded_len = n_blocks * block_size
    audio_padded = np.zeros(padded_len)
    audio_padded[:n_samples] = audio
    
    # Convert payload to bits
    payload_len = len(payload_bytes)
    header = np.array([payload_len], dtype=np.uint32).view(np.uint8)
    payload_with_header = header.tobytes() + payload_bytes

    payload_bits = np.unpackbits(
        np.frombuffer(payload_with_header, dtype=np.uint8)
    )
    
    # Extract magnitude and phase for all blocks
    magnitudes = []
    phases = []
    for b in range(n_blocks):
        block = audio_padded[b * block_size:(b + 1) * block_size]
        fft_block = np.fft.fft(block)
        mag = np.abs(fft_block)
        phase = np.angle(fft_block)
        magnitudes.append(mag)
        phases.append(phase)
    
    # Compute inter-block phase differences: Δφᵢ = φᵢ - φᵢ₋₁
    phase_diffs = []
    for b in range(1, n_blocks):
        diff = phases[b] - phases[b - 1]
        phase_diffs.append(diff)
    
    # Embed payload into block 0's phase
    # Map bits: 0 -> π/2, 1 -> -π/2
    # Only use half the frequencies due to conjugate symmetry
    max_payload_bits = block_size // 2
    payload_bits_truncated = payload_bits[:min(len(payload_bits), max_payload_bits)]
    
    phases_modified = [p.copy() for p in phases]
    for i, bit in enumerate(payload_bits_truncated, start=1):
        target_phase = np.pi / 2 if bit == 0 else -np.pi / 2
        phases_modified[0][i] = target_phase
    
    # Propagate original phase differences forward through remaining blocks
    for b in range(1, n_blocks):
        phases_modified[b] = phases_modified[b - 1] + phase_diffs[b - 1]
    
    # Enforce DFT symmetry for real-valued output
    for b in range(n_blocks):
        for k in range(1, block_size // 2):
            phases_modified[b][block_size - k] = -phases_modified[b][k]
        phases_modified[b][block_size // 2] = 0
    
    # Reconstruct via IFFT
    stego_padded = np.zeros(padded_len)
    for b in range(n_blocks):
        fft_reconstructed = magnitudes[b] * np.exp(1j * phases_modified[b])
        block_reconstructed = np.fft.ifft(fft_reconstructed).real
        stego_padded[b * block_size:(b + 1) * block_size] = block_reconstructed
    
    # Trim to original length
    stego_audio = stego_padded[:n_samples]
    
    # Normalize to prevent clipping
    max_val = np.max(np.abs(stego_audio))
    if max_val > 1.0:
        stego_audio = stego_audio / max_val
    
    return stego_audio

def phase_coding_decode(stego_audio, block_size=2048, expected_payload_bytes=None):
    """
    Classical phase coding extraction: read phase values from block 0.
    
    Args:
        stego_audio: numpy array of stego audio
        block_size: FFT block length (must match encoding)
        expected_payload_bytes: if known, use to terminate extraction early
    
    Returns:
        recovered_bytes: extracted payload
    """
    audio = np.asarray(stego_audio, dtype=np.float64)
    n_samples = len(audio)
    n_blocks = (n_samples + block_size - 1) // block_size
    
    # Pad to multiple of block_size
    padded_len = n_blocks * block_size
    audio_padded = np.zeros(padded_len)
    audio_padded[:n_samples] = audio
    
    # Extract phase from block 0
    block_0 = audio_padded[:block_size]
    fft_block_0 = np.fft.fft(block_0)
    phase_block_0 = np.angle(fft_block_0)
    
    # Extract bits by comparing phase to ±π/2
    max_payload_bits = block_size // 2
    extracted_bits = []
    
    for i in range(1, max_payload_bits):
        phase_val = phase_block_0[i]
        # Normalize phase to [-π, π]
        phase_val = np.angle(np.exp(1j * phase_val))
        
        # Determine which threshold is closer
        dist_to_pos = np.abs(phase_val - np.pi / 2)
        dist_to_neg = np.abs(phase_val + np.pi / 2)
        
        if dist_to_pos < dist_to_neg:
            extracted_bits.append(0)
        else:
            extracted_bits.append(1)
    
    # Convert bit array to bytes
    extracted_bits = np.array(extracted_bits[:max_payload_bits], dtype=np.uint8)
    
    # Pad to byte boundary
    remainder = len(extracted_bits) % 8
    if remainder != 0:
        extracted_bits = np.pad(extracted_bits, (0, 8 - remainder), mode='constant')
    
    # Pack bits into bytes
    extracted_bytes = np.packbits(extracted_bits)

    # read payload length from header
    payload_len = int(np.frombuffer(extracted_bytes[:4], dtype=np.uint32)[0])

    # extract payload
    payload = extracted_bytes[4:4+payload_len]

    return bytes(payload)

def phase_coding_capacity(block_size=2048):
    """Maximum payload capacity in bits for phase coding (block 0 only)."""
    return block_size // 2

def phase_coding_encode2(audio_data, payload_bytes, sample_rate, block_size=2048):

    audio = np.asarray(audio_data, dtype=np.float64)
    n_samples = len(audio)
    n_blocks = (n_samples + block_size - 1) // block_size

    padded_len = n_blocks * block_size
    audio_padded = np.zeros(padded_len)
    audio_padded[:n_samples] = audio

    # ----- HEADER -----
    payload_len = len(payload_bytes)
    header = np.array([payload_len], dtype=np.uint32).view(np.uint8)
    payload_with_header = header.tobytes() + payload_bytes

    payload_bits = np.unpackbits(
        np.frombuffer(payload_with_header, dtype=np.uint8)
    )

    # ----- MID FREQUENCY BINS (1–8 kHz) -----
    freqs = np.fft.fftfreq(block_size, 1 / sample_rate)

    mid_bins = np.where((freqs >= 1000) & (freqs <= 8000))[0]
    mid_bins = mid_bins[mid_bins < block_size // 2]

    bits_per_block = len(mid_bins)
    max_capacity = bits_per_block * n_blocks

    payload_bits = payload_bits[:max_capacity]

    # ----- FFT ANALYSIS -----
    magnitudes = []
    phases = []

    for b in range(n_blocks):

        block = audio_padded[b * block_size:(b + 1) * block_size]

        fft_block = np.fft.fft(block)

        magnitudes.append(np.abs(fft_block))
        phases.append(np.angle(fft_block))

    phases_modified = [p.copy() for p in phases]

    # ----- MULTI BLOCK EMBEDDING -----
    bit_idx = 0

    for b in range(n_blocks):

        for k in mid_bins:

            if bit_idx >= len(payload_bits):
                break

            bit = payload_bits[bit_idx]

            target_phase = np.pi / 2 if bit == 0 else -np.pi / 2

            phases_modified[b][k] = target_phase

            bit_idx += 1

    # ----- ENFORCE CONJUGATE SYMMETRY -----
    for b in range(n_blocks):

        for k in range(1, block_size // 2):
            phases_modified[b][block_size - k] = -phases_modified[b][k]

        phases_modified[b][block_size // 2] = 0

    # ----- IFFT RECONSTRUCTION -----
    stego_padded = np.zeros(padded_len)

    for b in range(n_blocks):

        fft_reconstructed = magnitudes[b] * np.exp(1j * phases_modified[b])

        block_reconstructed = np.fft.ifft(fft_reconstructed).real

        stego_padded[b * block_size:(b + 1) * block_size] = block_reconstructed

    stego_audio = stego_padded[:n_samples]

    max_val = np.max(np.abs(stego_audio))

    if max_val > 1:
        stego_audio = stego_audio / max_val

    return stego_audio

def phase_coding_decode2(stego_audio, sample_rate, block_size=2048):

    audio = np.asarray(stego_audio, dtype=np.float64)

    n_samples = len(audio)
    n_blocks = (n_samples + block_size - 1) // block_size

    padded_len = n_blocks * block_size

    audio_padded = np.zeros(padded_len)
    audio_padded[:n_samples] = audio

    # ----- MID FREQUENCY BINS -----
    freqs = np.fft.fftfreq(block_size, 1 / sample_rate)

    mid_bins = np.where((freqs >= 1000) & (freqs <= 8000))[0]
    mid_bins = mid_bins[mid_bins < block_size // 2]

    extracted_bits = []

    for b in range(n_blocks):

        block = audio_padded[b * block_size:(b + 1) * block_size]

        fft_block = np.fft.fft(block)

        phase_block = np.angle(fft_block)

        for k in mid_bins:

            phase_val = phase_block[k]

            phase_val = np.angle(np.exp(1j * phase_val))

            dist_pos = np.abs(phase_val - np.pi / 2)
            dist_neg = np.abs(phase_val + np.pi / 2)

            bit = 0 if dist_pos < dist_neg else 1

            extracted_bits.append(bit)

    extracted_bits = np.array(extracted_bits, dtype=np.uint8)

    remainder = len(extracted_bits) % 8

    if remainder != 0:
        extracted_bits = np.pad(extracted_bits, (0, 8 - remainder))

    extracted_bytes = np.packbits(extracted_bits)

    payload_len = int(np.frombuffer(extracted_bytes[:4], dtype=np.uint32)[0])

    payload = extracted_bytes[4:4 + payload_len]

    return bytes(payload)

def phase_coding_capacity2(sample_rate, block_size=2048):

    freqs = np.fft.fftfreq(block_size, 1 / sample_rate)

    mid_bins = np.where((freqs >= 1000) & (freqs <= 8000))[0]
    mid_bins = mid_bins[mid_bins < block_size // 2]

    return len(mid_bins)

def spectrogram_encode(image, n_fft=512, hop_length=None, sr=SR, max_freq=None):
    """
    Synthesize audio whose spectrogram displays the given image.

    Args:
        image:      PIL Image, path string, or Path to an image file.
        n_fft:      FFT size. Controls vertical (frequency) resolution.
        hop_length: Samples per time step. Defaults to n_fft // 4.
        sr:         Sample rate of output audio.
        max_freq:   If set, only use frequency bins up to this frequency (Hz).
                    Remaining upper bins are zero. Useful when the signal will
                    be frequency-shifted and must fit below Nyquist.

    Returns:
        audio:  np.ndarray (float64, normalized to [-1, 1])
        params: dict with n_fft, hop_length, image_shape for decode
    """
    # Load and convert to grayscale
    if isinstance(image, (str, Path)):
        image = Image.open(image).convert("L")
    elif isinstance(image, Image.Image):
        image = image.convert("L")

    hop_length = hop_length or n_fft // 4
    freq_bins = n_fft // 2 + 1

    # Determine how many frequency bins the image occupies
    if max_freq is not None:
        nyquist = sr / 2
        img_freq_bins = max(1, round(max_freq / nyquist * freq_bins))
        img_freq_bins = min(img_freq_bins, freq_bins)
    else:
        img_freq_bins = freq_bins

    # Resize image: scale so height = img_freq_bins, width preserves aspect ratio
    time_steps = round(image.width * img_freq_bins / image.height)
    image_resized = image.resize((time_steps, img_freq_bins), Image.LANCZOS)

    # Convert to numpy, normalize to [0, 1]
    pixels = np.array(image_resized, dtype=np.float64) / 255.0

    # Flip vertically: row 0 in numpy = DC (lowest freq),
    # but in the image row 0 = top. We want low freq at bottom of spectrogram.
    pixels = np.flipud(pixels)

    # Build STFT magnitude matrix: shape (freq_bins, time_steps)
    # Pad with zeros above img_freq_bins if needed
    if img_freq_bins < freq_bins:
        S = np.zeros((freq_bins, time_steps), dtype=np.float64)
        S[:img_freq_bins, :] = pixels
    else:
        S = pixels

    # Assign random phase for natural-sounding synthesis
    phase = np.random.uniform(-np.pi, np.pi, S.shape)
    stft_complex = S * np.exp(1j * phase)

    # Inverse STFT to get audio
    audio = librosa.istft(stft_complex, hop_length=hop_length, n_fft=n_fft)

    # Normalize to [-1, 1]
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    params = {
        "n_fft": n_fft,
        "hop_length": hop_length,
        "image_shape": (img_freq_bins, time_steps),
    }
    return audio, params

def spectrogram_decode(audio, n_fft=512, hop_length=None, image_shape=None):
    """
    Recover the hidden image from synthesized spectrogram audio.

    Args:
        audio:       np.ndarray of audio samples.
        n_fft:       FFT size (must match encode).
        hop_length:  Hop length (must match encode).
        image_shape: (height, width) to resize recovered image (optional).

    Returns:
        PIL Image (grayscale)
    """
    hop_length = hop_length or n_fft // 4

    # Compute STFT magnitude
    S = np.abs(librosa.stft(audio, n_fft=n_fft, hop_length=hop_length))

    # Normalize to [0, 255]
    S_max = S.max()
    if S_max > 0:
        S_norm = (S / S_max * 255).astype(np.uint8)
    else:
        S_norm = np.zeros_like(S, dtype=np.uint8)

    # Flip vertically to restore image orientation
    S_norm = np.flipud(S_norm)

    # STFT shape is (freq_bins, time_steps) = (height, width) — already correct for image
    img = Image.fromarray(S_norm, mode="L")

    if image_shape is not None:
        # image_shape is (freq_bins, time_steps) = (height, width)
        img = img.resize((image_shape[1], image_shape[0]), Image.LANCZOS)

    return img

def load_audio_stereo(file_path):
    """Load audio preserving stereo channels. Returns (np.ndarray shape (N,2), sr)."""
    audio, sr = sf.read(file_path)
    if audio.ndim == 1:
        raise ValueError(f"{file_path} is mono; stereo required for M/S encoding")
    if audio.shape[1] > 2:
        audio = audio[:, :2]
    return audio, sr

def frequency_shift(signal, shift_hz, sr):
    """Shift signal spectrum up by shift_hz using SSB modulation (analytic signal)."""
    analytic = hilbert(signal)
    t = np.arange(len(signal)) / sr
    shifted = np.real(analytic * np.exp(1j * 2 * np.pi * shift_hz * t))
    return shifted

def ms_camouflage_encode(stereo_audio, sr, image, n_fft=512,
                          freq_shift_hz=13000, gain_db=-35):
    """
    Hide image in the side channel of stereo audio via M/S encoding.

    Args:
        stereo_audio:  np.ndarray shape (N, 2) - stereo carrier
        sr:            sample rate
        image:         PIL Image or path
        n_fft:         FFT size for spectrogram painting
        freq_shift_hz: shift image-audio to this base frequency (Hz)
        gain_db:       attenuation in dB (e.g. -35)

    Returns:
        stego_stereo: np.ndarray shape (N, 2) - stego stereo audio
        params:       dict with encoding parameters for decode
    """
    L, R = stereo_audio[:, 0], stereo_audio[:, 1]
    M = (L + R) / 2
    S = (L - R) / 2

    # Generate image audio — limit bandwidth so it fits below Nyquist after shift
    available_bw = sr / 2 - freq_shift_hz
    img_audio, enc_params = spectrogram_encode(image, n_fft=n_fft, sr=sr, max_freq=available_bw)

    # Frequency-shift to ultrasonic range
    img_shifted = frequency_shift(img_audio, freq_shift_hz, sr)

    # Attenuate
    gain_linear = 10 ** (gain_db / 20)
    img_scaled = img_shifted * gain_linear

    # Match length to carrier
    if len(img_scaled) > len(S):
        img_scaled = img_scaled[:len(S)]
    elif len(img_scaled) < len(S):
        img_scaled = np.pad(img_scaled, (0, len(S) - len(img_scaled)))

    # Inject into side channel
    S_stego = S + img_scaled

    # Reconstruct stereo
    L_stego = M + S_stego
    R_stego = M - S_stego

    # Clip to prevent overflow
    L_stego = np.clip(L_stego, -1.0, 1.0)
    R_stego = np.clip(R_stego, -1.0, 1.0)

    stego_stereo = np.column_stack([L_stego, R_stego])

    params = {
        "enc_params": enc_params,
        "freq_shift_hz": freq_shift_hz,
        "gain_db": gain_db,
        "n_fft": n_fft,
    }
    return stego_stereo, params

def ms_camouflage_decode(stego_stereo, sr, n_fft=512, freq_shift_hz=13000,
                          hop_length=None, image_shape=None):
    """
    Extract hidden image from the side channel of stego stereo audio.

    Uses bandpass filtering around the target frequency range, then
    computes the STFT magnitude to recover pixel values.

    Returns:
        image:        PIL Image (grayscale) - recovered image
        side_channel: np.ndarray - isolated S channel for visualization
    """
    hop_length = hop_length or n_fft // 4

    # Extract side channel
    L, R = stego_stereo[:, 0], stego_stereo[:, 1]
    S_stego = (L - R) / 2

    # Bandpass filter around the hidden signal frequency range
    nyquist = sr / 2
    low = freq_shift_hz / nyquist
    high = min(freq_shift_hz + 6000, nyquist - 100) / nyquist
    high = min(high, 0.99)  # stay below Nyquist
    sos = butter(6, [low, high], btype="band", output="sos")
    S_filtered = sosfilt(sos, S_stego)

    # Compute STFT of the filtered side channel
    S_spec = np.abs(librosa.stft(S_filtered, n_fft=n_fft, hop_length=hop_length))

    # Normalize to [0, 255]
    S_max = S_spec.max()
    if S_max > 0:
        S_norm = (S_spec / S_max * 255).astype(np.uint8)
    else:
        S_norm = np.zeros_like(S_spec, dtype=np.uint8)

    S_norm = np.flipud(S_norm)
    img = Image.fromarray(S_norm, mode="L")

    if image_shape is not None:
        img = img.resize((image_shape[1], image_shape[0]), Image.LANCZOS)

    return img, S_stego

def echo_encode(audio, sr, bitstream, d0_ms=1.0, d1_ms=1.5, alpha=0.5, block_size=2048):
    """
    Echo hiding encoder: Embeds a binary payload by adding artificial echoes to the signal.
    Uses linear cross-fading of full-length echoed signals to prevent block-boundary artifacts.
    """
    d0_samples = int(sr * d0_ms / 1000)
    d1_samples = int(sr * d1_ms / 1000)
    
    # Generate two full echoed versions to avoid discontinuities
    y0 = np.copy(audio)
    y1 = np.copy(audio)
    
    y0[d0_samples:] += alpha * audio[:-d0_samples]
    y1[d1_samples:] += alpha * audio[:-d1_samples]
    
    # Generate mixer signal
    mixer = np.zeros_like(audio, dtype=float)
    
    for i, bit in enumerate(bitstream):
        start = i * block_size
        end = min((i + 1) * block_size, len(audio))
        if bit == 1:
            mixer[start:end] = 1.0
            
    # Apply smoothing to mixer (linear cross-fade at boundaries)
    fade_len = int(block_size * 0.1) # 10% overlap smoothing
    if fade_len > 0:
        window = scipy.signal.windows.hann(fade_len * 2)
        mixer_smoothed = scipy.signal.fftconvolve(mixer, window/np.sum(window), mode='same')
    else:
        mixer_smoothed = mixer
        
    y = y0 * (1 - mixer_smoothed) + y1 * mixer_smoothed
    return y

def echo_decode(stego, sr, num_bits, d0_ms=1.0, d1_ms=1.5, block_size=2048):
    """
    Echo hiding decoder: Extracts the bitstream from stego audio using 
    cepstral autocorrelation peak detection.
    """
    d0_samples = int(sr * d0_ms / 1000)
    d1_samples = int(sr * d1_ms / 1000)
    
    extracted_bits = []
    
    # We add a small tolerance for finding the peak in discrete sample index
    tol = 2
    
    for i in range(num_bits):
        start = i * block_size
        end = start + block_size
        if end > len(stego):
            break
            
        block = stego[start:end]
        
        # Apply window to block to reduce spectral leakage
        window = scipy.signal.windows.hann(len(block))
        block_windowed = block * window
        
        # Compute Real Cepstrum
        # 1. Power spectrum
        spectrum = np.abs(np.fft.fft(block_windowed))**2
        # 2. Log of spectrum
        log_spectrum = np.log(spectrum + 1e-10)
        # 3. IFFT -> real cepstrum
        cepstrum = np.real(np.fft.ifft(log_spectrum))
        
        # Autocorrelation of cepstrum (to find repeating peaks)
        autocorr = np.correlate(cepstrum, cepstrum, mode='full')
        autocorr = autocorr[len(autocorr)//2:] # keep positive lags
        
        # Determine peaks
        r0_start, r0_end = max(0, d0_samples - tol), min(len(autocorr), d0_samples + tol + 1)
        r1_start, r1_end = max(0, d1_samples - tol), min(len(autocorr), d1_samples + tol + 1)
        
        peak0 = np.max(autocorr[r0_start:r0_end]) if r0_start < r0_end else 0
        peak1 = np.max(autocorr[r1_start:r1_end]) if r1_start < r1_end else 0
        
        if peak1 > peak0:
            extracted_bits.append(1)
        else:
            extracted_bits.append(0)
            
    return extracted_bits

def steganalysis_echo(audio, sr, block_size=2048, threshold=0.01, max_delay_ms=10.0):
    """
    Steganalysis detector: Check cepstral autocorrelation for dominant short-quefrency peaks.
    max_delay_ms controls the upper quefrency search bound (must cover both d0 and d1).
    """
    num_blocks = len(audio) // block_size
    detection_count = 0
    max_search_samples = int(sr * max_delay_ms / 1000)
    min_search_samples = int(sr * 0.5 / 1000) # ignore 0-lag and very short delays
    
    blocks_to_check = min(num_blocks, 100)
    for i in range(blocks_to_check):
        block = audio[i*block_size : (i+1)*block_size]
        window = scipy.signal.windows.hann(len(block))
        
        spectrum = np.abs(np.fft.fft(block * window))**2
        log_spectrum = np.log(spectrum + 1e-10)
        cepstrum = np.real(np.fft.ifft(log_spectrum))
        
        autocorr = np.correlate(cepstrum, cepstrum, mode='full')
        autocorr = autocorr[len(autocorr)//2:]
        
        # Normalize autocorr
        autocorr = autocorr / (np.max(np.abs(autocorr)) + 1e-10)
        
        # Use absolute value — echo peaks can be negative
        peak = np.max(np.abs(autocorr[min_search_samples:max_search_samples]))
        if peak > threshold:
            detection_count += 1
            
    rate = detection_count / blocks_to_check
    return rate

def plot_cepstrum_autocorr(audio, stego, sr, block_size=2048, plot_range=None):
    """
    Plot the cepstrum autocorrelation for the first two blocks to visualize both echo peaks (d0 and d1).
    Typically, for ASCII strings, the first bit is '0' and the second is '1'.
    """
    import matplotlib.pyplot as plt
    
    window = scipy.signal.windows.hann(block_size)
    
    def get_autocorr(blk):
        spec = np.abs(np.fft.fft(blk * window))**2
        log_spec = np.log(spec + 1e-10)
        ceps = np.real(np.fft.ifft(log_spec))
        ac = np.correlate(ceps, ceps, mode='full')
        ac = ac[len(ac)//2:]
        return ac / (np.max(np.abs(ac)) + 1e-10)

    ac_orig = get_autocorr(audio[0:block_size])
    ac_stego0 = get_autocorr(stego[0:block_size]) # Usually encodes '0'
    ac_stego1 = get_autocorr(stego[block_size:block_size*2]) # Usually encodes '1'
    
    time_axis = np.arange(len(ac_orig)) / sr * 1000 # ms
    limit = plot_range if plot_range else 4.0
    plot_limit = int(sr * limit / 1000) # zoom in on 0 to 4 ms
    
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    
    # Original Audio
    ax[0].plot(time_axis[5:plot_limit], ac_orig[5:plot_limit], color='blue')
    ax[0].set_title("Autocorrelation (Clean, Block 0)")
    ax[0].set_xlabel("Quefrency (ms)")
    ax[0].set_ylabel("Normalized Amplitude")
    ax[0].grid(True, alpha=0.5)
    
    # Stego Audio Block 0 
    ax[1].plot(time_axis[5:plot_limit], ac_stego0[5:plot_limit], color='green')
    ax[1].set_title("Autocorrelation (Stego, Block 0)")
    ax[1].set_xlabel("Quefrency (ms)")
    ax[1].grid(True, alpha=0.5)
    
    # Stego Audio Block 1
    ax[2].plot(time_axis[5:plot_limit], ac_stego1[5:plot_limit], color='orange')
    ax[2].set_title("Autocorrelation (Stego, Block 1)")
    ax[2].set_xlabel("Quefrency (ms)")
    ax[2].grid(True, alpha=0.5)
    
    plt.tight_layout()
    plt.show()

def dsss_encode(audio_data, payload_bytes, key=42, chip_rate=1000, epsilon=0.01):
    """
    Direct Sequence Spread Spectrum (DSSS) Encoding.
    """
    audio = np.asarray(audio_data, dtype=np.float64)
    
    # Header: 32 bits (4 bytes) representing the payload length in bytes
    header = len(payload_bytes).to_bytes(4, byteorder="big")
    full_payload = header + payload_bytes
    
    # Convert payload to bits
    payload_bits = np.unpackbits(np.frombuffer(full_payload, dtype=np.uint8))
    
    if len(payload_bits) * chip_rate > len(audio):
        raise ValueError(f"Payload too large for carrier at given chip_rate. Need {len(payload_bits) * chip_rate} samples, but have {len(audio)}.")
        
    # Bipolar encoding: 0 -> -1, 1 -> 1
    bipolar_msg = np.where(payload_bits == 0, -1, 1)
    
    # Generate PRNG chip sequence
    if isinstance(key, str):
        key = int(hashlib.sha256(key.encode('utf-8')).hexdigest(), 16) % (2**32)
    np.random.seed(key)
    base_chip = np.random.choice([-1, 1], size=chip_rate)
    
    # Repeat the chip sequence for each bit to induce cyclostationarity
    chip_sequence = np.tile(base_chip, len(bipolar_msg))
    
    # Spreading: Multiply bipolar message by chip sequence
    spread_msg = np.repeat(bipolar_msg, chip_rate) * chip_sequence
    
    # Add spread signal to audio
    stego_audio = audio.copy()
    stego_audio[:len(spread_msg)] += epsilon * spread_msg
    
    # Ensure no clipping
    stego_audio = np.clip(stego_audio, -1.0, 1.0)
    
    return stego_audio

def dsss_decode(stego_audio, key=42, chip_rate=1000):
    """
    Direct Sequence Spread Spectrum (DSSS) Decoding via correlation.
    """
    audio = np.asarray(stego_audio, dtype=np.float64)
    
    # Re-generate the repeating chip sequence
    if isinstance(key, str):
        key = int(hashlib.sha256(key.encode('utf-8')).hexdigest(), 16) % (2**32)
    np.random.seed(key)
    base_chip = np.random.choice([-1, 1], size=chip_rate)
    
    extracted_bits = []
    
    # First, extract the header (4 bytes = 32 bits)
    header_bits_needed = 32
    if len(audio) < header_bits_needed * chip_rate:
        return b""
        
    for i in range(header_bits_needed):
        segment = audio[i*chip_rate : (i+1)*chip_rate]
        # Integrate (correlate) over the chip window
        correlation = np.sum(segment * base_chip)
        extracted_bits.append(1 if correlation > 0 else 0)
        
    extracted_bits_arr = np.array(extracted_bits, dtype=np.uint8)
    header_bytes = np.packbits(extracted_bits_arr)
    payload_len = int.from_bytes(header_bytes.tobytes(), byteorder='big')
    
    # Extract the payload based on header length
    payload_bits_needed = payload_len * 8
    total_bits_needed = header_bits_needed + payload_bits_needed
    
    if payload_len == 0 or len(audio) < total_bits_needed * chip_rate:
        return b"" # Header corrupted or audio truncated
        
    for i in range(header_bits_needed, total_bits_needed):
        segment = audio[i*chip_rate : (i+1)*chip_rate]
        correlation = np.sum(segment * base_chip)
        extracted_bits.append(1 if correlation > 0 else 0)
        
    extracted_bits_arr = np.array(extracted_bits, dtype=np.uint8)
    full_bytes = np.packbits(extracted_bits_arr).tobytes()
    
    return full_bytes[4:]

def cyclic_autocorrelation(audio, max_lag=2000):
    """
    Simplified cyclic autocorrelation to detect periodic chip sequence.
    We apply a simple difference filter (high-pass) to remove the dominant 
    low-frequency carrier audio, revealing the hidden high-frequency chip sequence.
    """
    # Use a chunk of audio to speed up computation
    chunk_size = min(len(audio), 44100)
    chunk = audio[:chunk_size]
    
    # Simple high-pass filter to remove cover audio
    chunk = np.diff(chunk)
    
    # Mean center
    chunk = chunk - np.mean(chunk)
    
    # Autocorrelation
    n = len(chunk)
    autocorr = np.correlate(chunk, chunk, mode='full')[n-1:]
    
    # Normalize
    if autocorr[0] == 0:
        return autocorr[:max_lag]
    
    autocorr = autocorr / autocorr[0]
    return autocorr[:max_lag]

def preprocess_wav(input_path, output_path):
    y, sr = librosa.load(input_path, sr=44100, mono=False)
    
    if y.ndim == 1:
        y = np.vstack((y, y))

    num_samples = y.shape[1]
    trim_len = (num_samples // 1152) * 1152
    y = y[:, :trim_len]
    
    padding = np.zeros((2, 1152))
    y = np.hstack((y, padding))

    sf.write(output_path, y.T, 44100, subtype='PCM_16')
    return output_path

def run_demonstration(cover_name, message, bitrate=320):
    stego = Steganography(quiet=True)
    wav_path = str(AUDIO_DIR / cover_name)

    processed_wav_path = str("./tmp/temp_standardized.wav")

    clean_mp3 = str("./tmp/clean_reference.mp3")
    stego_mp3 = str("./tmp/stego_output.mp3")
    result_txt = "./tmp/extracted.txt"
    preprocess_wav(wav_path, processed_wav_path)
    # --- Clean MP3 ---
    stego.encode_wav_to_mp3(processed_wav_path, clean_mp3, bitrate)

    # --- Stego MP3 ---
    stego.hide_message(clean_mp3, stego_mp3, message)

    # --- Extract Message ---
    stego.reveal_massage(stego_mp3, result_txt)

    file = open(result_txt, "r")
    extracted_msg = file.read()
    print(f"Extracted message: {extracted_msg}")
    file.close()

    # --- Reversibility ---
    cleaned_mp3 = str("./tmp/cleaned_trace.mp3")
    stego.clear_file(stego_mp3, cleaned_mp3)

    return clean_mp3, stego_mp3, cleaned_mp3

def compare_audio(path1, path2):
    # Wczytujemy bez resamplingu (sr=None)
    y1, sr1 = librosa.load(path1, sr=None)
    y2, sr2 = librosa.load(path2, sr=None)
    
    correlation = np.correlate(y1[:20000], y2[:20000], mode='full')
    offset = np.argmax(correlation) - (len(y2[:20000]) - 1)
    
    if offset > 0:
        y1 = y1[offset:]
    elif offset < 0:
        y2 = y2[-offset:]
    min_len = min(len(y1), len(y2))
    y1, y2 = y1[:min_len], y2[:min_len]
    residual = y1 - y2
    
    snr = 10 * np.log10(np.sum(y1**2) / (np.sum(residual**2) + 1e-10))
    
    return y1, y2, residual, snr

def play_audio_results(clean_path, stego_path, cleaned_path):
    print("### 1. Clean MP3 (Reference) ###")
    y_clean, sr = librosa.load(clean_path, sr=None)
    display(Audio(data=y_clean, rate=sr))
    
    print("### 2. Stego MP3 (With Hidden Message) ###")
    y_stego, _ = librosa.load(stego_path, sr=sr)
    display(Audio(data=y_stego, rate=sr))
    
    print("### 3. Cleaned MP3 (After Reversibility) ###")
    y_cleaned, _ = librosa.load(cleaned_path, sr=sr)
    display(Audio(data=y_cleaned, rate=sr))

## UI Setup

### Algorithm Configs

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
import numpy as np
import os
from pathlib import Path

# Provide access to the image and audio directories
AUDIO_FILES = [f.name for f in AUDIO_DIR.glob("*.wav")] if AUDIO_DIR.exists() else []
IMAGE_FILES = [f.name for f in IMG_DIR.glob("*.png")] if IMG_DIR.exists() else []

# Define configurations and parameters for each algorithm
ALGO_CONFIGS = {
    "LSB": {
        "payload": "text",
        "requires_cover": True,
        "params": {
            "n_bits": widgets.IntSlider(value=1, min=1, max=8, description="n_bits:")
        }
    },
    "Phase Coding (Classical)": {
        "payload": "text",
        "requires_cover": True,
        "params": {
            "block_size": widgets.Dropdown(options=[512, 1024, 2048, 4096, 8192], value=2048, description="block_size:")
        }
    },
    "Phase Coding (Improved)": {
        "payload": "text",
        "requires_cover": True,
        "params": {
            "block_size": widgets.Dropdown(options=[512, 1024, 2048, 4096, 8192], value=2048, description="block_size:")
        }
    },
    "Echo Hiding": {
        "payload": "text",
        "requires_cover": True,
        "params": {
            "d0_ms": widgets.FloatSlider(value=1.0, min=0.5, max=10.0, step=0.1, description="d0_ms:"),
            "d1_ms": widgets.FloatSlider(value=1.5, min=0.5, max=10.0, step=0.1, description="d1_ms:"),
            "alpha": widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description="alpha:"),
            "block_size": widgets.Dropdown(options=[512, 1024, 2048, 4096, 8192], value=2048, description="block_size:")
        }
    },
    "DSSS": {
        "payload": "text",
        "requires_cover": True,
        "params": {
            "chip_rate": widgets.IntSlider(value=2000, min=100, max=5000, step=100, description="chip_rate:"),
            "epsilon": widgets.FloatSlider(value=0.015, min=0.001, max=0.05, step=0.001, description="epsilon:"),
            "key": widgets.Text(value="hunter2", description="key:")
        }
    },
    "Spectrogram Painting": {
        "payload": "image",
        "requires_cover": False,  # Synthesizes audio entirely
        "params": {
            "n_fft": widgets.Dropdown(options=[256, 512, 1024, 2048, 4096], value=512, description="n_fft:")
        }
    },
    "M/S Camouflage": {
        "payload": "image",
        "requires_cover": True,
        "params": {
            "n_fft": widgets.Dropdown(options=[256, 512, 1024, 2048, 4096], value=512, description="n_fft:"),
            "freq_shift_hz": widgets.IntSlider(value=13000, min=8000, max=20000, step=500, description="freq_shift_hz:"),
            "gain_db": widgets.IntSlider(value=-35, min=-60, max=-10, step=1, description="gain_db:")
        }
    },
    "MP3Stego": {
        "payload": "text",
        "requires_cover": True,
        "params": {
            "bitrate": widgets.Dropdown(options=[64, 128, 192, 256, 320], value=320, description="bitrate (kbps):")
        }
    }
}

# --- Shared UI Elements ---
w_algo = widgets.Dropdown(options=list(ALGO_CONFIGS.keys()), value="LSB", description="Algorithm:", layout=widgets.Layout(width="400px"), style={"description_width": "120px"})
w_audio = widgets.Dropdown(options=AUDIO_FILES, description="Cover audio:", layout=widgets.Layout(width="400px"), style={"description_width": "120px"})
w_image = widgets.Dropdown(options=IMAGE_FILES, description="Payload Image:", layout=widgets.Layout(width="400px"), style={"description_width": "120px"})
w_message = widgets.Textarea(value="Hello, this is a secret message!", placeholder="Enter text to hide...", description="Message:", layout=widgets.Layout(width="560px", height="80px"), style={"description_width": "120px"})
w_button = widgets.Button(description="Encode & Decode", button_style="primary", icon="play", layout=widgets.Layout(width="200px", height="36px"))
w_out = widgets.Output()
w_params_container = widgets.VBox()

# --- Dynamic UI Updates ---
def update_ui(*args):
    config = ALGO_CONFIGS[w_algo.value]
    
    # Update visible parameters
    w_params_container.children = tuple(config["params"].values())
    
    # Update payload input (Text vs Image)
    if config["payload"] == "text":
        w_message.layout.display = "flex"
        w_image.layout.display = "none"
    else:
        w_message.layout.display = "none"
        w_image.layout.display = "flex"
        
    # Toggle cover audio requirement
    if config["requires_cover"]:
        w_audio.layout.display = "flex"
    else:
        w_audio.layout.display = "none"

w_algo.observe(update_ui, 'value')
update_ui() # Initialize

### Encode/Decode Pipeline Logic

In [6]:
def run_pipeline(btn):
    btn.disabled = True
    try:
        algo_name = w_algo.value
        config = ALGO_CONFIGS[algo_name]
        audio_name = w_audio.value
        params = {k: v.value for k, v in config["params"].items()}
        
        with w_out:
            clear_output(wait=True)
            print(f"--- Running Pipeline: {algo_name} ---")
            
            # 1. Load Cover Audio (If needed)
            cover_audio = None
            sr = SR
            if config["requires_cover"]:
                try:
                    if algo_name == "M/S Camouflage":
                        cover_audio, sr = load_audio_stereo(AUDIO_DIR / audio_name)
                        print(f"Loaded Stereo Cover: {audio_name} @ {sr}Hz")
                    else:
                        cover_audio, sr = load_audio(AUDIO_DIR / audio_name)
                        print(f"Loaded Mono Cover: {audio_name} @ {sr}Hz")
                except Exception as e:
                    print(f"ERROR loading audio: {e}")
                    return
                    
            # 2. Prepare Payload
            payload = None
            if config["payload"] == "text":
                message = w_message.value.strip()
                if not message:
                    print("ERROR: Please enter a message.")
                    return
                payload = text_to_bytes(message)
                print(f"Payload: Text ({len(payload)} bytes)")
            elif config["payload"] == "image":
                image_name = w_image.value
                if not image_name:
                    print("ERROR: Please select an image.")
                    return
                payload = IMG_DIR / image_name
                print(f"Payload: Image ({image_name})")
                
            # 3. Encode & Decode Mapping
            stego_audio = None
            decoded_data = None
            
            try:
                if algo_name == "LSB":
                    stego_audio = lsb_encode(cover_audio, payload, n_bits=params["n_bits"])
                    decoded_data = lsb_decode(stego_audio, n_bits=params["n_bits"])
                    
                elif algo_name == "Phase Coding (Classical)":
                    stego_audio = phase_coding_encode(cover_audio, payload, block_size=params["block_size"])
                    decoded_data = phase_coding_decode(stego_audio, block_size=params["block_size"])
                    
                elif algo_name == "Phase Coding (Improved)":
                    stego_audio = phase_coding_encode2(cover_audio, payload, sr, block_size=params["block_size"])
                    decoded_data = phase_coding_decode2(stego_audio, sr, block_size=params["block_size"])
                    
                elif algo_name == "Echo Hiding":
                    payload_bits = np.unpackbits(np.frombuffer(payload, dtype=np.uint8))
                    stego_audio = echo_encode(cover_audio, sr, payload_bits, d0_ms=params["d0_ms"], d1_ms=params["d1_ms"], alpha=params["alpha"], block_size=params["block_size"])
                    decoded_bits = echo_decode(stego_audio, sr, len(payload_bits), d0_ms=params["d0_ms"], d1_ms=params["d1_ms"], block_size=params["block_size"])
                    valid = (len(decoded_bits) // 8) * 8
                    decoded_data = np.packbits(decoded_bits[:valid]).tobytes()
                    
                elif algo_name == "DSSS":
                    stego_audio = dsss_encode(cover_audio, payload, key=params["key"], chip_rate=params["chip_rate"], epsilon=params["epsilon"])
                    decoded_data = dsss_decode(stego_audio, key=params["key"], chip_rate=params["chip_rate"])
                    
                elif algo_name == "Spectrogram Painting":
                    stego_audio, enc_params = spectrogram_encode(payload, n_fft=params["n_fft"], sr=sr)
                    decoded_data = spectrogram_decode(stego_audio, n_fft=params["n_fft"], hop_length=enc_params["hop_length"], image_shape=enc_params["image_shape"])
                    
                elif algo_name == "M/S Camouflage":
                    stego_audio, enc_params = ms_camouflage_encode(cover_audio, sr, payload, n_fft=params["n_fft"], freq_shift_hz=params["freq_shift_hz"], gain_db=params["gain_db"])
                    decoded_data, _ = ms_camouflage_decode(stego_audio, sr, n_fft=params["n_fft"], freq_shift_hz=params["freq_shift_hz"], hop_length=enc_params["enc_params"]["hop_length"], image_shape=enc_params["enc_params"]["image_shape"])
                    
                elif algo_name == "MP3Stego":
                    # Uses the run_demonstration logic from section 6
                    print("Running MP3Stego demonstration wrapper...")
                    clean_p, stego_p, cleaned_p = run_demonstration(audio_name, w_message.value.strip(), bitrate=params["bitrate"])
                    print("MP3Stego uses files on disk. Generating visual differences:")
                    y_clean, y_stego, res, snr_val = compare_audio(clean_p, cleaned_p)
                    print(f"SNR (Clean vs Cleaned): {snr_val:.2f} dB")
                    play_audio_results(clean_p, stego_p, cleaned_p)
                    return # MP3Stego handles its own output
                    
            except Exception as e:
                print(f"\n[!] Pipeline Error: {e}")
                import traceback
                traceback.print_exc()
                return
                
            print("Encoding and Decoding complete.")
            
            # 4. Display Results
            print("\n--- Results ---")
            if config["requires_cover"]:
                orig_trimmed = cover_audio[:len(stego_audio)]
                # Flatten to calculate SNR for stereo signals
                snr_val = snr(orig_trimmed.flatten(), stego_audio.flatten())
                print(f"SNR (Original vs Stego): {snr_val:.2f} dB")
            
            if config["payload"] == "text":
                orig_bits = np.unpackbits(np.frombuffer(payload, dtype=np.uint8))
                dec_bits = np.unpackbits(np.frombuffer(decoded_data, dtype=np.uint8))
                n = min(len(orig_bits), len(dec_bits))
                if n > 0:
                    ber_val = np.mean(orig_bits[:n] != dec_bits[:n]) * 100
                    print(f"BER: {ber_val:.2f}%")
                    
                decoded_text = bytes_to_text(decoded_data)
                print(f"\nOriginal : {w_message.value.strip()[:100]}...")
                print(f"Decoded  : {decoded_text[:100]}...")
                if decoded_text.strip() == w_message.value.strip():
                    print("Status: Perfect reconstruction! ✅")
                else:
                    print("Status: Imperfect reconstruction or truncation. ⚠️")
                    
            elif config["payload"] == "image":
                fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                axes[0].imshow(np.array(Image.open(payload).convert("L")), cmap="gray")
                axes[0].set_title("Original Image")
                axes[0].axis("off")
                axes[1].imshow(np.array(decoded_data), cmap="gray")
                axes[1].set_title("Recovered Image")
                axes[1].axis("off")
                plt.tight_layout()
                plt.show()
    
            if not config["requires_cover"]:
                print("\nStego Audio Playback:")
                # Display Audio correctly depending on shape
                if stego_audio.ndim > 1:
                    display(Audio(data=stego_audio.T, rate=sr))
                else:
                    display(Audio(data=stego_audio, rate=sr))
                
            if config["requires_cover"]:
                plt.close('all')
                display_audio_comparison(cover_audio, stego_audio, sr, title=f"{algo_name} - {audio_name}")
    finally:
        btn.disabled = False

w_button._click_handlers.callbacks.clear()
w_button.on_click(run_pipeline)

# The Playground
Where the fun happens.

In [7]:
display(widgets.VBox([
    widgets.HTML("<h2>Audio Steganography Playground</h2>"),
    widgets.HTML("<p>Select an algorithm, adjust its specific parameters dynamically, and process the payload.</p>"),
    w_algo,
    w_audio,
    w_image,
    w_message,
    widgets.HTML("<hr><h4>Algorithm Parameters</h4>"),
    w_params_container,
    widgets.HTML("<br>"),
    w_button,
    widgets.HTML("<hr>"),
    w_out
]))